In [1]:
"""
Phase 3 remote inference server — paste into a Kaggle notebook.

Hosts three HTTP endpoints behind one ngrok tunnel:
    POST /sdxl      text prompt → still image (Stable Diffusion XL Turbo)
    POST /svd       still image → ~3s ambient-motion clip (Stable Video Diffusion)
    POST /lipsync   image|video + audio → talking-head clip (Wav2Lip)
    GET  /health    quick liveness check + model load status

Usage:
    1. Open a new notebook on kaggle.com/code/new.
    2. Settings → Accelerator → "GPU T4 x2" (free).
    3. Settings → Internet → "On".
    4. Add Secrets:
       - NGROK_AUTH_TOKEN (free token from ngrok.com).
       - HF_TOKEN (free token from huggingface.co/settings/tokens) — used
         for higher-rate-limit downloads of SDXL-Turbo + SVD.
    5. Paste the contents of this file into a single code cell, or split
       at the `# %%` markers into one cell each.
    6. Run all. Wait for "Public URL: https://<random>.ngrok-free.app".
    7. Copy that URL into your local .env as KAGGLE_ENDPOINT=...
    8. Run `python main.py --prompt "..."` locally — Phase 3 will hit
       these endpoints and skip the local fallback.

VRAM budget on T4 (16 GB): all three models use diffusers'
`enable_model_cpu_offload()` so only the active pipeline sits in VRAM.
Idle weights swap back to system RAM. Peak GPU usage is whichever
endpoint is currently running.

Sessions time out after ~9 hours; re-run the notebook and update the
URL in your .env when that happens.
"""

# %% Cell 1 — system + Python deps
#
# We deliberately don't pin numpy / numba / opencv. Kaggle ships modern
# versions of those (numpy 2.x, numba 0.60+) that conflict with Wav2Lip's
# original 2020 dependencies. Pinning them down provokes the "ResolutionImpossible"
# wall and breaks the rest of Kaggle's preinstalled stack (cuml, jax, etc.).
# Wav2Lip's `np.complex` / `np.float` use is patched at runtime in cell 5 instead.
#
!apt-get -y -qq install ffmpeg
!pip install -q --upgrade pip
!pip install -q fastapi uvicorn pyngrok python-multipart httpx nest-asyncio
!pip install -q "diffusers>=0.27.2" "transformers>=4.41.0" "accelerate>=0.30.1" safetensors torchvision
!pip install -q "librosa>=0.10.1" "soundfile>=0.12.1"

# (Optional) authenticate huggingface_hub so SDXL-Turbo + SVD-XT downloads
# get higher rate limits. The token is read from a Kaggle Secret named HF_TOKEN.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("HF authenticated")
except Exception as e:
    print(f"HF login skipped — {e}")


# %% Cell 2 — clone Wav2Lip + download checkpoints
!cd /kaggle/working && git clone --depth 1 https://github.com/Rudrabha/Wav2Lip.git
!mkdir -p /kaggle/working/Wav2Lip/checkpoints /kaggle/working/Wav2Lip/face_detection/detection/sfd

# Wav2Lip GAN checkpoint (~400MB) — community mirror.
!wget -q -O /kaggle/working/Wav2Lip/checkpoints/wav2lip_gan.pth \
    https://huggingface.co/numz/wav2lip/resolve/main/wav2lip_gan.pth

# S3FD face detector (~100MB).
!wget -q -O /kaggle/working/Wav2Lip/face_detection/detection/sfd/s3fd.pth \
    https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.5 MB/s eta 0:00:0000:01
HF authenticated
Cloning into 'Wav2Lip'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 57 (delta 3), reused 39 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 442.23 KiB | 8.67 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [2]:


import os, subprocess
ckpt = "/kaggle/working/Wav2Lip/checkpoints/wav2lip_gan.pth"
if os.path.exists(ckpt):
    os.remove(ckpt)

# Mirrors in order of reliability — the first one that returns >400 MB wins.
mirrors = [
    "https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth",
    "https://huggingface.co/numz/wav2lip/resolve/main/wav2lip_gan.pth",
    "https://huggingface.co/camenduru/Wav2Lip/resolve/main/wav2lip_gan.pth",
]

for url in mirrors:
    print(f"→ Trying {url}", flush=True)
    rc = subprocess.call(
        ["wget", "-q", "--tries=2", "-O", ckpt, url],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    size_mb = os.path.getsize(ckpt) / 1024 / 1024 if os.path.exists(ckpt) else 0
    print(f"   downloaded: {size_mb:.1f} MB, exit code {rc}")
    if size_mb > 400:
        break
    print("   too small — trying next")

# Final verification
import torch
try:
    state = torch.load(ckpt, map_location="cpu", weights_only=False)
    n_tensors = len(state.get("state_dict", state))
    print(f"\n✅ Checkpoint loads OK — {n_tensors} tensors, {os.path.getsize(ckpt)/1024/1024:.1f} MB")
except Exception as e:
    print(f"\n❌ STILL BROKEN: {type(e).__name__}: {e}")

→ Trying https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth
   downloaded: 415.6 MB, exit code 0

✅ Checkpoint loads OK — 352 tensors, 415.6 MB


In [3]:

# %% Cell 3 — set up paths + the rest is plain Python.
import base64
import io
import os
import shutil
import subprocess
import sys
import tempfile
import threading
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from fastapi import FastAPI, HTTPException
from PIL import Image
from pydantic import BaseModel

# Kaggle working directory layout:
KAGGLE_ROOT = Path("/kaggle/working")
WAV2LIP_DIR = KAGGLE_ROOT / "Wav2Lip"
sys.path.insert(0, str(WAV2LIP_DIR))


In [4]:
# %% Cell 4 — GPU plan + SVD pipeline (LAZY: loaded on first /svd request)
#
# Kaggle's "GPU T4 x2" gives us two 16 GB T4s. We give each diffusers pipeline
# its own GPU so they never compete for VRAM:
#     cuda:0 — SVD-XT  (~9 GB)        + Wav2Lip subprocess (~2 GB) = ~11 GB
#     cuda:1 — SDXL-Turbo (~7 GB)
# When only one GPU is available we serialise the two via _GPU_LOCK and
# park each pipeline on CPU between calls (slower, but safe).

_SVD_PIPE = None
_SVD_LOCK = threading.Lock()
_GPU_LOCK = threading.Lock()    # used only when the two share one GPU

_NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
SVD_DEVICE  = "cuda:0" if _NUM_GPUS >= 1 else "cpu"
SDXL_DEVICE = "cuda:1" if _NUM_GPUS >= 2 else SVD_DEVICE
_SHARE_GPU = (SVD_DEVICE == SDXL_DEVICE) and _NUM_GPUS >= 1
print(f"[GPU plan] SVD={SVD_DEVICE}, SDXL={SDXL_DEVICE}, share={_SHARE_GPU}")


def _unload_models_to_cpu():
    """
    Free GPU memory between pipeline runs.

    Important: we deliberately do NOT call `.to("cpu")` on the loaded
    pipelines.

    - SDXL is loaded with `torch_dtype=torch.float16`. fp16 pipelines can't
      actually run on CPU, and diffusers' `.to("cpu")` round-trip leaves
      sub-modules in a half-migrated state — when the next `.to(cuda:1)`
      call comes, some buffers stay on CPU and inference raises
      `Cannot generate a cpu tensor from a generator of type cuda`.

    - SVD uses `enable_model_cpu_offload()`; accelerate already keeps its
      weights paged out at rest, and calling `.to("cpu")` on a hooked
      pipeline corrupts the hooks.

    What we actually need is to recycle the GPU *cache* on cuda:0 (where
    SVD intermediate latents and Wav2Lip subprocess remnants accumulate).
    SDXL on cuda:1 has the GPU to itself and isn't the OOM source.
    """
    import gc

    # Drop Python references to any temp tensors held in module-local state.
    gc.collect()

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            try:
                with torch.cuda.device(i):
                    torch.cuda.synchronize()
                    torch.cuda.empty_cache()
                    torch.cuda.reset_peak_memory_stats()
                    free_mb = int(torch.cuda.mem_get_info()[0] / 1024 / 1024)
                    used_mb = int(torch.cuda.memory_allocated() / 1024 / 1024)
                    print(f"[unload] cuda:{i} → {free_mb} MB free, {used_mb} MB allocated")
            except Exception as exc:
                print(f"[unload] cuda:{i} cleanup failed: {exc!r}")
    print("[unload] complete (pipelines kept on GPU, cache cleared)")


def _force_unload_pipelines():
    """
    Hard-reset: drop the loaded pipeline references entirely so the next
    request reloads from disk. Use this only if `_unload_models_to_cpu()`
    isn't enough — e.g., you suspect a leak in pipeline state itself.
    Costs ~30 s of model reload on the next request. Not called automatically.
    """
    import gc
    global _SDXL_PIPE, _SVD_PIPE
    _SDXL_PIPE = None
    _SVD_PIPE = None
    gc.collect()
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                torch.cuda.empty_cache()
    print("[force-unload] dropped pipeline refs — next request will reload")


def get_svd_pipeline():
    global _SVD_PIPE
    if _SVD_PIPE is not None:
        return _SVD_PIPE
    with _SVD_LOCK:
        if _SVD_PIPE is None:
            from diffusers import StableVideoDiffusionPipeline

            pipe = StableVideoDiffusionPipeline.from_pretrained(
                "stabilityai/stable-video-diffusion-img2vid-xt",
                torch_dtype=torch.float16,
                variant="fp16",
            )
            # SVD-XT at 1024×576×25 frames needs ~14 GB peak during VAE decode
            # — barely fits on a T4 even alone, and Wav2Lip subprocesses share
            # this GPU. Use model_cpu_offload so weights page in/out per stage,
            # dropping peak VRAM to ~5 GB.
            #
            # NOTE: model_cpu_offload is incompatible with manual `.to(device)`
            # — accelerate's hooks expect the model to start on CPU. So we
            # don't move the pipeline ourselves here.
            gpu_id = 0 if SVD_DEVICE.startswith("cuda") else None
            if gpu_id is not None:
                pipe.enable_model_cpu_offload(gpu_id=gpu_id)
            else:
                pipe = pipe.to("cpu")
            _SVD_PIPE = pipe
    return _SVD_PIPE


def run_svd(image_path: str, num_frames: int = 25, motion_strength: int = 127) -> str:
    """Run SVD-XT and return the path to a freshly-rendered MP4."""
    pipe = get_svd_pipeline()
    image = Image.open(image_path).convert("RGB").resize((1024, 576))

    # _SHARE_GPU only matters when SVD and SDXL collide on cuda:0 (single-GPU
    # sessions). Even with 2 GPUs we still want to free cuda:0 between runs
    # so the Wav2Lip subprocess can allocate. model_cpu_offload does that
    # automatically; we additionally call empty_cache() after.
    if _SHARE_GPU:
        with _GPU_LOCK:
            try:
                frames = _svd_inference(pipe, image, num_frames, motion_strength)
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    else:
        try:
            frames = _svd_inference(pipe, image, num_frames, motion_strength)
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Write frames to MP4 via imageio (already pulled in by diffusers).
    import imageio.v2 as imageio  # noqa: WPS433

    out = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    fps = 6
    imageio.mimwrite(out, [np.array(f) for f in frames], fps=fps,
                     codec="libx264", quality=8)
    return out


def _svd_inference(pipe, image, num_frames, motion_strength):
    # Steps trade quality for time. 25 = published default (~6.5 min on T4
    # with model_cpu_offload + decode_chunk_size=1). 8 = ~2.5 min, noticeably
    # noisier but fits under ngrok's free-tier ~60 s silent-request limit
    # for the synchronous /svd response.
    with torch.inference_mode():
        return pipe(
            image,
            num_frames=num_frames,
            num_inference_steps=int(os.getenv("SVD_STEPS", "8")),
            motion_bucket_id=motion_strength,
            noise_aug_strength=0.02,
            decode_chunk_size=2,   # 1→2 cuts decode wall-time roughly in half
                                   # while still fitting in T4's ~10 GB free VRAM
        ).frames[0]

[GPU plan] SVD=cuda:0, SDXL=cuda:1, share=False


In [5]:
# %% Cell 4b — SDXL-Turbo pipeline (LAZY: loaded on first /sdxl request)

_SDXL_PIPE = None
_SDXL_LOCK = threading.Lock()


def get_sdxl_pipeline():
    global _SDXL_PIPE
    if _SDXL_PIPE is not None:
        return _SDXL_PIPE
    with _SDXL_LOCK:
        if _SDXL_PIPE is None:
            # Import StableDiffusionXLPipeline directly instead of going through
            # AutoPipelineForText2Image — the auto pipeline transitively imports
            # every pipeline diffusers ships (HunyuanDiT, etc.) and one of those
            # needs `MT5Tokenizer`, which transformers only exposes when
            # `sentencepiece` is installed. Direct import sidesteps the chain.
            from diffusers import StableDiffusionXLPipeline

            pipe = StableDiffusionXLPipeline.from_pretrained(
                "stabilityai/sdxl-turbo",
                torch_dtype=torch.float16,
                variant="fp16",
            )
            if hasattr(pipe, "safety_checker"):
                pipe.safety_checker = None
            # When SDXL has its own GPU, park there permanently. When sharing
            # with SVD, stay on CPU and swap to GPU only inside run_sdxl.
            pipe = pipe.to("cpu" if _SHARE_GPU else SDXL_DEVICE)
            _SDXL_PIPE = pipe
    return _SDXL_PIPE


def run_sdxl(prompt: str, *, seed: int = 0, width: int = 1280, height: int = 720,
             steps: int = 1, guidance: float = 0.0) -> bytes:
    """Generate one SDXL-Turbo image and return its PNG bytes."""
    pipe = get_sdxl_pipeline()

    if _SHARE_GPU:
        with _GPU_LOCK:
            try:
                pipe.to(SDXL_DEVICE)
                img = _sdxl_inference(pipe, prompt, seed, steps, guidance)
            finally:
                pipe.to("cpu")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    else:
        # Dedicated GPU — pipeline already lives there.
        img = _sdxl_inference(pipe, prompt, seed, steps, guidance)

    img = img.convert("RGB").resize((width, height))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def _sdxl_inference(pipe, prompt, seed, steps, guidance):
    # SDXL-Turbo is trained at 512x512. Render square then resize to the
    # requested aspect — sharper than asking the model for 1280x720.
    gen_device = SDXL_DEVICE if torch.cuda.is_available() else "cpu"
    gen = torch.Generator(device=gen_device).manual_seed(int(seed))
    with torch.inference_mode():
        result = pipe(
            prompt=prompt,
            num_inference_steps=int(steps),
            guidance_scale=float(guidance),
            generator=gen,
            width=512,
            height=512,
        )
    return result.images[0]


In [6]:
# %% Cell 5 — Wav2Lip wrapper (PRELOAD: small enough not to bother with lazy)

_WAV2LIP_READY = False


def _patch_wav2lip_source():
    """
    Rewrite Wav2Lip's `audio.py` so its positional `librosa.filters.mel(...)`
    call is keyword-style. librosa 0.10+ made those args keyword-only.
    Idempotent — safe to call repeatedly.
    """
    audio_py = WAV2LIP_DIR / "audio.py"
    if not audio_py.exists():
        return
    src = audio_py.read_text(encoding="utf-8")
    patched = src.replace(
        "librosa.filters.mel(hp.sample_rate, hp.n_fft,",
        "librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,",
    )
    if patched != src:
        audio_py.write_text(patched, encoding="utf-8")
        print("Patched Wav2Lip/audio.py for librosa 0.10+ keyword args")


def _ensure_wav2lip():
    """numpy / librosa compatibility shims + path validation + source patch."""
    global _WAV2LIP_READY

    # The source-file patch is idempotent — always run it so re-running this
    # cell after a previous successful run still applies the fix.
    _patch_wav2lip_source()

    if _WAV2LIP_READY:
        return

    # numpy 2.x removed `np.complex` / `np.float` / `np.int`.
    # Wav2Lip's vendored face_detection still uses them.
    for alias, real in (("complex", complex), ("float", float),
                        ("int", int), ("bool", bool)):
        if not hasattr(np, alias):
            setattr(np, alias, real)

    # librosa 0.10 made `filters.mel` keyword-only. Wav2Lip's audio.py calls it
    # positionally: librosa.filters.mel(sr, n_fft, n_mels=..., fmin=..., fmax=...)
    # Wrap it so both calling conventions work.
    try:
        import librosa  # type: ignore
        _orig_mel = librosa.filters.mel
        import inspect
        params = inspect.signature(_orig_mel).parameters
        if list(params.keys())[:2] != ["sr", "n_fft"] or any(
            p.kind == inspect.Parameter.KEYWORD_ONLY for p in list(params.values())[:2]
        ):
            def _mel_compat(*args, **kwargs):
                if len(args) >= 1 and "sr" not in kwargs:
                    kwargs["sr"] = args[0]
                if len(args) >= 2 and "n_fft" not in kwargs:
                    kwargs["n_fft"] = args[1]
                return _orig_mel(**kwargs)
            librosa.filters.mel = _mel_compat  # type: ignore
    except Exception:
        pass

    ckpt = WAV2LIP_DIR / "checkpoints" / "wav2lip_gan.pth"
    if not ckpt.exists():
        raise RuntimeError("wav2lip_gan.pth not found — run cell 2 first")

    # Validate the checkpoint loads cleanly. Truncated downloads produce a
    # file that exists, looks plausibly large, but torch.load() hits EOFError
    # mid-pickle 4 minutes into your first /lipsync request. Catch it now.
    size_mb = ckpt.stat().st_size / 1024 / 1024
    if size_mb < 400:
        raise RuntimeError(
            f"wav2lip_gan.pth is only {size_mb:.1f} MB (expected ~436 MB) — "
            f"re-run the wget in cell 2 (the download was truncated)"
        )
    try:
        torch.load(str(ckpt), map_location="cpu", weights_only=False)
    except Exception as exc:
        raise RuntimeError(
            f"wav2lip_gan.pth failed to deserialize: {type(exc).__name__}: {exc}\n"
            f"Re-download from a different mirror in cell 2."
        )

    _WAV2LIP_READY = True


In [7]:
# %% Cell 6 — FastAPI app
class SDXLRequest(BaseModel):
    prompt: str
    seed: int = 0
    width: int = 1280
    height: int = 720
    steps: int = 1
    guidance: float = 0.0


class SDXLResponse(BaseModel):
    image_b64: str


class SVDRequest(BaseModel):
    image_b64: str
    motion_strength: int = 127
    num_frames: int = 25


class SVDResponse(BaseModel):
    clip_mp4_b64: str
    fps: int
    frames: int


class LipSyncRequest(BaseModel):
    face_b64: str
    face_kind: str = "image"   # "image" or "video"
    audio_b64: str             # 16kHz mono WAV (the local client resamples)


class LipSyncResponse(BaseModel):
    mp4_b64: str


app = FastAPI(title="Phase 3 remote inference")


def _b64_to_tmp(b64: str, suffix: str) -> str:
    fd, path = tempfile.mkstemp(suffix=suffix)
    with os.fdopen(fd, "wb") as f:
        f.write(base64.b64decode(b64))
    return path


def _file_to_b64(path: str) -> str:
    return base64.b64encode(Path(path).read_bytes()).decode("ascii")


@app.get("/health")
def health():
    loaded = []
    if _SDXL_PIPE is not None:
        loaded.append("sdxl")
    if _SVD_PIPE is not None:
        loaded.append("svd")
    if _WAV2LIP_READY:
        loaded.append("wav2lip")
    vram_per_gpu = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            vram_per_gpu.append(int(torch.cuda.memory_allocated(i) / 1024 / 1024))
    return {
        "models_loaded": loaded,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "vram_used_mb": int(torch.cuda.memory_allocated() / 1024 / 1024) if torch.cuda.is_available() else 0,
        "vram_per_gpu_mb": vram_per_gpu,
    }


@app.post("/unload")
def unload_endpoint():
    """
    Free GPU memory by paging all loaded pipelines back to CPU and clearing
    CUDA caches. Pipelines reload to GPU lazily on the next inference request.
    Use this between pipeline runs to avoid VRAM accumulation without
    restarting the Kaggle kernel.
    """
    before_mb = int(torch.cuda.memory_allocated() / 1024 / 1024) if torch.cuda.is_available() else 0
    _unload_models_to_cpu()
    after_mb = int(torch.cuda.memory_allocated() / 1024 / 1024) if torch.cuda.is_available() else 0
    return {"freed_mb": before_mb - after_mb, "vram_used_mb": after_mb}


@app.post("/sdxl", response_model=SDXLResponse)
def sdxl_endpoint(req: SDXLRequest):
    try:
        png_bytes = run_sdxl(
            req.prompt, seed=req.seed,
            width=req.width, height=req.height,
            steps=req.steps, guidance=req.guidance,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"SDXL error: {exc}")
    return SDXLResponse(image_b64=base64.b64encode(png_bytes).decode("ascii"))


@app.post("/svd", response_model=SVDResponse)
def svd_endpoint(req: SVDRequest):
    img_path = _b64_to_tmp(req.image_b64, ".png")
    try:
        out_mp4 = run_svd(img_path, num_frames=req.num_frames,
                          motion_strength=req.motion_strength)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"SVD error: {exc}")
    finally:
        Path(img_path).unlink(missing_ok=True)
    try:
        return SVDResponse(clip_mp4_b64=_file_to_b64(out_mp4), fps=6, frames=req.num_frames)
    finally:
        Path(out_mp4).unlink(missing_ok=True)


@app.post("/lipsync", response_model=LipSyncResponse)
def lipsync_endpoint(req: LipSyncRequest):
    face_suffix = ".mp4" if req.face_kind == "video" else ".png"
    face_path = _b64_to_tmp(req.face_b64, face_suffix)
    audio_path = _b64_to_tmp(req.audio_b64, ".wav")
    try:
        out_mp4 = run_wav2lip(face_path, audio_path)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Wav2Lip error: {exc}")
    finally:
        Path(face_path).unlink(missing_ok=True)
        Path(audio_path).unlink(missing_ok=True)
    try:
        return LipSyncResponse(mp4_b64=_file_to_b64(out_mp4))
    finally:
        Path(out_mp4).unlink(missing_ok=True)


In [8]:
# %% Cell 7 — start uvicorn + open ngrok tunnel
#
# Jupyter/Kaggle already has a running asyncio loop, so calling `uvicorn.run`
# directly raises "asyncio.run() cannot be called from a running event loop".
# We work around it by:
#   1. nest_asyncio.apply()  → allows nested loops
#   2. running uvicorn.Server.serve() inside a daemon thread with its own loop
#      → the cell returns immediately so /health checks etc. can run after.

import asyncio  # noqa: E402
import threading  # noqa: E402

import nest_asyncio  # noqa: E402
import uvicorn  # noqa: E402
from pyngrok import conf, ngrok  # noqa: E402

# Pull token from Kaggle Secrets — the cell errors loudly if not set.
from kaggle_secrets import UserSecretsClient  # type: ignore  # noqa: E402

ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
conf.get_default().auth_token = ngrok_token

# Preload Wav2Lip helpers (cheap) so /health reflects readiness.
_ensure_wav2lip()

import time  # noqa: E402

# If a previous run of this cell is still serving on port 8000, shut it down
# cleanly. We stash references to the server + thread on the module's globals
# so re-runs can find and stop them.
_PREV_SERVER = globals().get("_KAGGLE_SERVER")
_PREV_THREAD = globals().get("_KAGGLE_SERVER_THREAD")
if _PREV_SERVER is not None:
    print("Stopping previous uvicorn server...")
    _PREV_SERVER.should_exit = True
    if _PREV_THREAD is not None and _PREV_THREAD.is_alive():
        # Bumped from 5 s — drain in-flight inference cleanly (Wav2Lip + SVD
        # are long-running) before we touch model state.
        _PREV_THREAD.join(timeout=30)
    # Linux keeps the socket in TIME_WAIT briefly after release.
    time.sleep(2)

    # Free GPU memory while the new server is binding — pipelines move back
    # to their GPU on the next inference request.
    try:
        _unload_models_to_cpu()
    except NameError:
        # Cell 4 hasn't been (re-)run in this session; nothing to unload.
        pass

# Same with the ngrok tunnel — disconnect any zombies before we open a new one.
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass

public_url = ngrok.connect(8000, "http")
print("=" * 70)
print(f"  Public URL:  {public_url.public_url}")
print(f"  Set this in your local .env:  KAGGLE_ENDPOINT={public_url.public_url}")
print("=" * 70)

nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning", loop="asyncio")
server = uvicorn.Server(config)


def _serve():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())


_thread = threading.Thread(target=_serve, daemon=True)
_thread.start()

# Stash refs so the next re-run of this cell can find and stop us.
_KAGGLE_SERVER = server
_KAGGLE_SERVER_THREAD = _thread

# Wait until the server reports it's ready before the cell returns.
for _ in range(50):
    if server.started:
        break
    time.sleep(0.1)
print("Server running in background thread — endpoints are live.")

Patched Wav2Lip/audio.py for librosa 0.10+ keyword args
  Public URL:  https://fernlike-khalil-unadeptly.ngrok-free.dev
  Set this in your local .env:  KAGGLE_ENDPOINT=https://fernlike-khalil-unadeptly.ngrok-free.dev
Server running in background thread — endpoints are live.
